In [2]:
import transformers

print("Transformers:", transformers.__version__)

Transformers: 4.57.6


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("BERT tokenizer loaded successfully!")

BERT tokenizer loaded successfully!


In [5]:
text = "Artificial Intelligence is changing the world."

tokens = tokenizer.tokenize(text)

print("Original text:", text)
print("Tokens:", tokens)

Original text: Artificial Intelligence is changing the world.
Tokens: ['artificial', 'intelligence', 'is', 'changing', 'the', 'world', '.']


In [6]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token IDs:", token_ids)

Tokens: ['artificial', 'intelligence', 'is', 'changing', 'the', 'world', '.']
Token IDs: [7976, 4454, 2003, 5278, 1996, 2088, 1012]


In [7]:
encoded = tokenizer(
    "Artificial Intelligence is changing the world."
)

print(encoded)

{'input_ids': [101, 7976, 4454, 2003, 5278, 1996, 2088, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [8]:
sentences = [
    "I love Artificial Intelligence.",
    "Deep Learning is interesting."
]

encoded = tokenizer(
    sentences,
    padding=True,
    return_tensors="tf"
)

print(encoded)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


{'input_ids': <tf.Tensor: shape=(2, 7), dtype=int32, numpy=
array([[ 101, 1045, 2293, 7976, 4454, 1012,  102],
       [ 101, 2784, 4083, 2003, 5875, 1012,  102]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(2, 7), dtype=int32, numpy=
array([[0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(2, 7), dtype=int32, numpy=
array([[1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1]], dtype=int32)>}


In [9]:
sentences = [
    "I love AI.",
    "Artificial Intelligence is changing the world."
]

encoded = tokenizer(
    sentences,
    padding=True,
    return_tensors="tf"
)

print("Input IDs:\n", encoded["input_ids"].numpy())
print("Attention Mask:\n", encoded["attention_mask"].numpy())

Input IDs:
 [[ 101 1045 2293 9932 1012  102    0    0    0]
 [ 101 7976 4454 2003 5278 1996 2088 1012  102]]
Attention Mask:
 [[1 1 1 1 1 1 0 0 0]
 [1 1 1 1 1 1 1 1 1]]


In [10]:
from transformers import AutoModel

bert_model = AutoModel.from_pretrained("bert-base-uncased")

print("BERT model loaded successfully!")

BERT model loaded successfully!


In [11]:
print("Number of layers:", bert_model.config.num_hidden_layers)
print("Hidden size:", bert_model.config.hidden_size)
print("Attention heads:", bert_model.config.num_attention_heads)
print("Vocabulary size:", bert_model.config.vocab_size)

Number of layers: 12
Hidden size: 768
Attention heads: 12
Vocabulary size: 30522


In [12]:
import torch

text = "Artificial Intelligence is changing the world."

inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[ 101, 7976, 4454, 2003, 5278, 1996, 2088, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [13]:
with torch.no_grad():
    outputs = bert_model(**inputs)

print(outputs.last_hidden_state.shape)

torch.Size([1, 9, 768])


In [14]:
cls_embedding = outputs.last_hidden_state[:, 0, :]

print("CLS embedding shape:", cls_embedding.shape)
print("First 10 values:", cls_embedding[0, :10])

CLS embedding shape: torch.Size([1, 768])
First 10 values: tensor([ 0.0704,  0.3369, -0.0227, -0.1167, -0.4364, -0.7451,  0.8188,  0.9011,
         0.0569, -0.6164])


In [15]:
sentences = [
    "I love machine learning.",
    "I hate machine learning."
]

inputs = tokenizer(
    sentences,
    padding=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = bert_model(**inputs)

embeddings = outputs.last_hidden_state[:, 0, :]

print("Embedding shape:", embeddings.shape)

Embedding shape: torch.Size([2, 768])


In [16]:
import torch.nn.functional as F

similarity = F.cosine_similarity(
    embeddings[0].unsqueeze(0),
    embeddings[1].unsqueeze(0)
)

print("Cosine Similarity:", similarity.item())

Cosine Similarity: 0.9756622314453125


### Hugging Face Pipelines - Sentiment Analysis

In [20]:
import torch
print("PyTorch:", torch.__version__)
print("Available:", torch.cuda.is_available())

PyTorch: 2.15.0+cpu
Available: False


In [22]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt",
    device=-1
)

print("Sentiment pipeline loaded successfully!")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


Sentiment pipeline loaded successfully!


In [ ]:
# Testing the sentiment analysis pipeline with a sample text
result = sentiment_analyzer(
    "I absolutely love learning Artificial Intelligence!"
)

print(result)

[{'label': 'POSITIVE', 'score': 0.9997465014457703}]


In [24]:
result = sentiment_analyzer(
    "I hate this experience. It was terrible and disappointing."
)

print(result)

[{'label': 'NEGATIVE', 'score': 0.999767005443573}]


### Fine-tuning

In [26]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    local_files_only=True
)

print("Model and tokenizer loaded successfully!")

Model and tokenizer loaded successfully!


In [27]:
texts = [
    "I love this product.",
    "This is an amazing experience.",
    "The quality is excellent.",
    "I am very happy with it.",
    "I hate this product.",
    "This was a terrible experience.",
    "The quality is very poor.",
    "I am disappointed with it."
]

labels = [1, 1, 1, 1, 0, 0, 0, 0]

print("Number of reviews:", len(texts))
print("Number of labels:", len(labels))

Number of reviews: 8
Number of labels: 8


In [28]:
encoded_data = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

print("Input IDs shape:", encoded_data["input_ids"].shape)
print("Attention Mask shape:", encoded_data["attention_mask"].shape)

Input IDs shape: torch.Size([8, 9])
Attention Mask shape: torch.Size([8, 9])


In [29]:
import torch
from torch.utils.data import TensorDataset, DataLoader

labels_tensor = torch.tensor(labels)

dataset = TensorDataset(
    encoded_data["input_ids"],
    encoded_data["attention_mask"],
    labels_tensor
)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

print("Total batches:", len(dataloader))

Total batches: 2


In [30]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

epochs = 2

for epoch in range(epochs):
    total_loss = 0

    for input_ids, attention_mask, batch_labels in dataloader:

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=batch_labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)

    print(f"Epoch {epoch + 1}, Loss: {average_loss:.4f}")

Epoch 1, Loss: 0.0003
Epoch 2, Loss: 0.0002


In [31]:
# Testing the fine-tuned model with new sentences
model.eval()

test_reviews = [
    "The service was wonderful and I enjoyed it.",
    "This was the worst purchase I ever made."
]

test_inputs = tokenizer(
    test_reviews,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**test_inputs)

predictions = torch.argmax(outputs.logits, dim=1)

for review, prediction in zip(test_reviews, predictions):
    sentiment = "POSITIVE" if prediction.item() == 1 else "NEGATIVE"
    print(review, "→", sentiment)

The service was wonderful and I enjoyed it. → POSITIVE
This was the worst purchase I ever made. → NEGATIVE


In [32]:
model.save_pretrained("my_finetuned_sentiment_model")
tokenizer.save_pretrained("my_finetuned_sentiment_model")

print("Fine-tuned model saved successfully!")

Fine-tuned model saved successfully!
